# 07 — Full-gene marker DE before annotation

This notebook reproduces the **article-defining historical marker-DE analysis**
on the canonical 18-cluster backbone produced by notebook `06`, while avoiding
the in-memory copies that are unsafe on the current Windows / 16-GB-RAM system.

## Historical scientific semantics preserved

The archived executed notebook used:

- full-gene expression: **15,176 genes**;
- canonical Leiden grouping: `leiden_scvi_main`;
- canonical `r = 0.60` partition: **18 clusters**;
- raw counts from `layers["counts"]`;
- `scanpy.pp.normalize_total(target_sum=1e4)`;
- `scanpy.pp.log1p`;
- `scanpy.tl.rank_genes_groups`;
- `method="wilcoxon"`;
- one-vs-rest (`reference="rest"`);
- `corr_method="benjamini-hochberg"`;
- `tie_correct=True`;
- `pts=True`;
- `rankby_abs=False`;
- all 15,176 genes ranked.

Archived execution checkpoints:

- all-gene DE table: `(273168, 11)`;
- top-100 up/down table: `(3600, 12)`;
- top-20 up table: `(360, 12)`;
- filtered table (`padj < 0.05`, `abs(logFC) >= 0.25`): `(192882, 11)`;
- historical cluster-0 top five: `PTPRZ1`, `BCAN`, `OLIG1`, `NOVA1`, `PTN`;
- marker-panel genes: 79/81 present; missing `MS4A1` and `JCHAIN`.

## Memory-safe implementation

Scanpy's public `rank_genes_groups` implementation does not accept backed matrix
types. Therefore the full `351427 × 15176` normalized matrix is never loaded
into RAM here.

Instead this notebook:

1. validates the canonical notebook-06 H5AD with metadata-only HDF5 reads;
2. creates the historical log-normalized `X` sequentially in a temporary H5AD;
3. simultaneously partitions normalized sparse entries into small gene-block
   cache files on NVMe scratch;
4. runs the **same Scanpy Wilcoxon calculation** on one gene block at a time;
5. performs Benjamini-Hochberg correction globally over all 15,176 genes per
   cluster, matching the historical whole-matrix semantics;
6. reconstructs the usual Scanpy ranking tables and stores them in the final
   marker-ready H5AD;
7. checkpoints both normalization and DE so an interruption can resume.

No cluster assignments are recomputed in this notebook.

**Windows/SciPy compatibility:** cached gene-block memmap fields are materialized as writable arrays before constructing each temporary CSR matrix because Scanpy performs an in-place `eliminate_zeros()` check.


## Expected historical checkpoints

The canonical input must contain exactly:

```text
351,427 cells
15,176 genes
18 clusters at leiden_scvi_main / r=0.60
cluster min/max/median = 105 / 61,646 / 14,336.5
```

The notebook stops rather than silently changing the historical analysis if
these structural invariants are not met.

In [1]:
# =========================
# 0) IMPORTS / PATHS / PARAMETERS
# =========================

import os
import gc
import json
import math
import hashlib
import shutil
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt

from statsmodels.stats.multitest import multipletests

try:
    from anndata.io import read_elem, write_elem
except ImportError:
    from anndata._io.specs import read_elem, write_elem


warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

sc.settings.verbosity = 2
sc.settings.set_figure_params(
    dpi=120,
    facecolor="white",
    frameon=True,
)

RANDOM_STATE = 42
np.random.seed(
    RANDOM_STATE
)


def _find_repository_root(start=None):
    start = (
        Path.cwd()
        if start is None
        else Path(start)
    )
    start = start.expanduser().resolve()

    for candidate in (
        start,
        *start.parents,
    ):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        _find_repository_root(),
    )
).expanduser().resolve()

INTEGRATED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "integrated"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
)

for directory in (
    INTEGRATED_DIR,
    METADATA_DIR,
    FIGURES_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Canonical article branch from notebook 06.
FULLGENE_INPUT_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad"
)

MARKER_READY_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_DE_ready.h5ad"
)

MARKER_READY_BUILDING_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_DE_ready.building.h5ad"
)


def _default_scratch_root():
    configured = os.environ.get(
        "GLIOMA_MAIN07_SCRATCH_ROOT"
    )

    if configured:
        return (
            Path(configured)
            .expanduser()
            .resolve()
        )

    if (
        os.name == "nt"
        and Path("E:/").exists()
    ):
        return Path(
            r"E:\glioma-cnv-scratch\main_analysis_07"
        )

    return (
        PROJECT_DIR
        / ".scratch"
        / "main_analysis_07"
    ).resolve()


SCRATCH_ROOT = _default_scratch_root()
BLOCK_CACHE_DIR = (
    SCRATCH_ROOT
    / "normalized_gene_blocks"
)
DE_STATE_DIR = (
    SCRATCH_ROOT
    / "de_state"
)

for directory in (
    SCRATCH_ROOT,
    BLOCK_CACHE_DIR,
    DE_STATE_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


NORMALIZATION_PROGRESS_JSON = (
    SCRATCH_ROOT
    / "normalization_progress.json"
)

DE_PROGRESS_JSON = (
    DE_STATE_DIR
    / "de_progress.json"
)


# Historical analysis parameters.
N_TOP_DIRECTION = 100
N_TOP_PLOT_PER_CLUSTER = 3

MAIN_CLUSTER_KEY = (
    "leiden_scvi_main"
)
MAIN_CLUSTER_ORIGINAL_KEY = (
    "leiden_scvi_HVG8000_r0_6"
)
REFINEMENT_CLUSTER_KEY = (
    "leiden_scvi_HVG8000_r0_7"
)

DATASET_COL = "core_dataset"
CONDITION_COL = "condition"
PATIENT_COL = "patient_id"
SAMPLE_COL = "sample_id"

EXPECTED_N_CELLS = 351_427
EXPECTED_N_GENES = 15_176
EXPECTED_N_CLUSTERS = 18

EXPECTED_CLUSTER_MIN = 105
EXPECTED_CLUSTER_MAX = 61_646
EXPECTED_CLUSTER_MEDIAN = 14_336.5

TARGET_SUM = np.float32(
    1e4
)

# 256 genes keeps each in-memory Scanpy block small while allowing
# Scanpy itself to use its historical internal ~28-gene Wilcoxon chunks.
GENE_BLOCK_SIZE = 256

# Sequential HDF5 read block. Adjust only for performance, not science.
MAX_NNZ_PER_ROW_BLOCK = 8_000_000

# One compact structured record per normalized non-zero entry.
# 4-byte row + 2-byte local column + 4-byte float value = 10 bytes.
BLOCK_RECORD_DTYPE = np.dtype(
    [
        ("row", "<i4"),
        ("col", "<u2"),
        ("val", "<f4"),
    ],
    align=False,
)


# Outputs.
DE_READINESS_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_readiness_report.tsv"
)

CLUSTER_QC_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_cluster_QC_summary.tsv"
)

CLUSTER_COMPOSITION_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_cluster_composition_summary.tsv"
)

MARKER_PANEL_PRESENCE_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_marker_panel_presence.tsv"
)

MISSING_MARKER_PANEL_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_missing_marker_panel_genes.tsv"
)

DE_ALL_TSV = (
    METADATA_DIR
    / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_all_genes.tsv"
)

DE_TOP100_UP_DOWN_TSV = (
    METADATA_DIR
    / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top100_up_down.tsv"
)

DE_TOP20_UP_TSV = (
    METADATA_DIR
    / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top20_up.tsv"
)

DE_FILTERED_TSV = (
    METADATA_DIR
    / "marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_filtered_padj005_logfc025.tsv"
)

DE_REPRO_AUDIT_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_historical_reproducibility_audit.tsv"
)

DE_OUTPUT_MANIFEST_TSV = (
    METADATA_DIR
    / "marker_DE_HVG8000_r0_6_output_manifest.tsv"
)

RANK_KEY = (
    "rank_genes_groups_leiden_scvi_main_HVG8000_r0_6_wilcoxon"
)

MARKER_FIG_DIR = (
    FIGURES_DIR
    / "marker_DE_HVG8000_r0_6_pre_annotation"
)
MARKER_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

sc.settings.figdir = str(
    MARKER_FIG_DIR
)


# Historical executed checkpoints.
HISTORICAL_ALL_DE_SHAPE = (
    273_168,
    11,
)

HISTORICAL_TOP100_SHAPE = (
    3_600,
    12,
)

HISTORICAL_TOP20_SHAPE = (
    360,
    12,
)

HISTORICAL_FILTERED_SHAPE = (
    192_882,
    11,
)

HISTORICAL_CLUSTER0_TOP5 = [
    "PTPRZ1",
    "BCAN",
    "OLIG1",
    "NOVA1",
    "PTN",
]

HISTORICAL_CLUSTER0_TOP5_SCORES = np.array(
    [
        307.749878,
        288.052246,
        280.849701,
        271.248901,
        269.065308,
    ],
    dtype=np.float64,
)


def write_tsv_replace(
    dataframe,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    dataframe.to_csv(
        path,
        sep="\t",
        index=False,
    )

    print(
        f"[SAVED] {path} "
        f"shape={dataframe.shape}"
    )


def write_json_atomic(
    payload,
    path,
):
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
        )

    os.replace(
        temporary,
        path,
    )


def hash_strings(
    values,
):
    digest = hashlib.sha256()

    for value in values:
        digest.update(
            str(value).encode(
                "utf-8"
            )
        )
        digest.update(
            b"\0"
        )

    return digest.hexdigest()


def natural_cluster_key(value):
    text = str(value)
    return (
        0,
        int(text),
    ) if text.isdigit() else (
        1,
        text,
    )


print(
    "PROJECT_DIR:",
    PROJECT_DIR,
)

print(
    "Input:",
    FULLGENE_INPUT_H5AD,
    "| exists:",
    FULLGENE_INPUT_H5AD.exists(),
)

print(
    "Scratch:",
    SCRATCH_ROOT,
)

PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
Input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad | exists: True
Scratch: E:\glioma-cnv-scratch\main_analysis_07


In [2]:
# =========================
# 1) METADATA-ONLY INPUT VALIDATION
# =========================

if not FULLGENE_INPUT_H5AD.exists():
    raise FileNotFoundError(
        FULLGENE_INPUT_H5AD
    )


with h5py.File(
    FULLGENE_INPUT_H5AD,
    "r",
) as handle:

    if "obs" not in handle:
        raise KeyError(
            "Input H5AD is missing /obs."
        )

    if "var" not in handle:
        raise KeyError(
            "Input H5AD is missing /var."
        )

    if (
        "layers" not in handle
        or "counts" not in handle["layers"]
    ):
        raise KeyError(
            "Input H5AD is missing layers['counts']."
        )

    counts_group = (
        handle[
            "layers"
        ][
            "counts"
        ]
    )

    counts_encoding = (
        counts_group.attrs.get(
            "encoding-type",
            ""
        )
    )

    if isinstance(
        counts_encoding,
        bytes,
    ):
        counts_encoding = (
            counts_encoding.decode(
                "utf-8"
            )
        )

    counts_shape = tuple(
        int(x)
        for x
        in counts_group.attrs[
            "shape"
        ]
    )

    counts_nnz = int(
        counts_group[
            "data"
        ].shape[0]
    )

    counts_dtype = str(
        counts_group[
            "data"
        ].dtype
    )

    obs = read_elem(
        handle[
            "obs"
        ]
    )

    var = read_elem(
        handle[
            "var"
        ]
    )

    x_umap = (
        np.asarray(
            handle[
                "obsm"
            ][
                "X_umap"
            ],
            dtype=np.float32,
        )
        if (
            "obsm" in handle
            and "X_umap"
            in handle["obsm"]
        )
        else None
    )

    x_scvi_shape = (
        tuple(
            int(x)
            for x
            in handle[
                "obsm"
            ][
                "X_scVI"
            ].shape
        )
        if (
            "obsm" in handle
            and "X_scVI"
            in handle["obsm"]
        )
        else None
    )

    sample_n = min(
        counts_nnz,
        1_000_000,
    )

    count_sample = np.asarray(
        counts_group[
            "data"
        ][
            :sample_n
        ]
    )


print(
    "counts encoding:",
    counts_encoding,
)

print(
    "counts shape:",
    counts_shape,
)

print(
    "counts nnz:",
    counts_nnz,
)

print(
    "counts dtype:",
    counts_dtype,
)

print(
    "obs rows:",
    len(obs),
)

print(
    "var rows:",
    len(var),
)

print(
    "X_scVI shape:",
    x_scvi_shape,
)

print(
    "X_umap shape:",
    None
    if x_umap is None
    else x_umap.shape,
)


if counts_encoding != "csr_matrix":
    raise AssertionError(
        "Historical marker-DE implementation "
        "expects CSR raw counts."
    )

if counts_shape != (
    EXPECTED_N_CELLS,
    EXPECTED_N_GENES,
):
    raise AssertionError(
        "Unexpected full-gene count shape: "
        f"{counts_shape}"
    )

if len(obs) != EXPECTED_N_CELLS:
    raise AssertionError(
        "Unexpected cell count."
    )

if len(var) != EXPECTED_N_GENES:
    raise AssertionError(
        "Unexpected gene count."
    )

if not var.index.is_unique:
    raise AssertionError(
        "Full-gene var_names are not unique."
    )

if MAIN_CLUSTER_KEY not in obs.columns:
    if MAIN_CLUSTER_ORIGINAL_KEY in obs.columns:
        obs[
            MAIN_CLUSTER_KEY
        ] = (
            obs[
                MAIN_CLUSTER_ORIGINAL_KEY
            ]
            .astype(str)
            .astype("category")
        )
    else:
        raise KeyError(
            "Canonical main cluster key is missing."
        )


required_obs = [
    MAIN_CLUSTER_KEY,
    DATASET_COL,
    CONDITION_COL,
    PATIENT_COL,
    SAMPLE_COL,
]

missing_required = [
    column
    for column
    in required_obs
    if column not in obs.columns
]

if missing_required:
    raise KeyError(
        "Missing required obs columns: "
        f"{missing_required}"
    )


for column in required_obs:
    obs[column] = (
        obs[column]
        .astype(str)
        .fillna(
            "unknown"
        )
        .astype(
            "category"
        )
    )


cluster_counts = (
    obs[
        MAIN_CLUSTER_KEY
    ]
    .astype(str)
    .value_counts()
)


if cluster_counts.shape[0] != EXPECTED_N_CLUSTERS:
    raise AssertionError(
        "Canonical clustering is not 18 clusters."
    )

if int(
    cluster_counts.min()
) != EXPECTED_CLUSTER_MIN:
    raise AssertionError(
        "Canonical minimum cluster size changed."
    )

if int(
    cluster_counts.max()
) != EXPECTED_CLUSTER_MAX:
    raise AssertionError(
        "Canonical maximum cluster size changed."
    )

if not np.isclose(
    float(
        cluster_counts.median()
    ),
    EXPECTED_CLUSTER_MEDIAN,
):
    raise AssertionError(
        "Canonical median cluster size changed."
    )


if not np.isfinite(
    count_sample
).all():
    raise ValueError(
        "Count sample contains non-finite values."
    )

if np.any(
    count_sample < 0
):
    raise ValueError(
        "Count sample contains negative values."
    )

if not np.allclose(
    count_sample,
    np.round(
        count_sample
    ),
):
    raise ValueError(
        "Count sample is not integer-like."
    )


input_fingerprint = {
    "source_path":
        str(
            FULLGENE_INPUT_H5AD.resolve()
        ),

    "source_size":
        int(
            FULLGENE_INPUT_H5AD.stat().st_size
        ),

    "source_mtime_ns":
        int(
            FULLGENE_INPUT_H5AD.stat().st_mtime_ns
        ),

    "shape":
        [
            EXPECTED_N_CELLS,
            EXPECTED_N_GENES,
        ],

    "nnz":
        counts_nnz,

    "obs_names_sha256":
        hash_strings(
            obs.index.astype(str)
        ),

    "var_names_sha256":
        hash_strings(
            var.index.astype(str)
        ),

    "main_cluster_sha256":
        hash_strings(
            obs[
                MAIN_CLUSTER_KEY
            ].astype(str)
        ),
}


readiness = pd.DataFrame(
    [
        {
            "metric":
                "input_h5ad",
            "value":
                str(
                    FULLGENE_INPUT_H5AD
                ),
        },
        {
            "metric":
                "n_cells",
            "value":
                EXPECTED_N_CELLS,
        },
        {
            "metric":
                "n_genes",
            "value":
                EXPECTED_N_GENES,
        },
        {
            "metric":
                "main_cluster_key",
            "value":
                MAIN_CLUSTER_KEY,
        },
        {
            "metric":
                "n_clusters",
            "value":
                EXPECTED_N_CLUSTERS,
        },
        {
            "metric":
                "counts_layer_present",
            "value":
                True,
        },
        {
            "metric":
                "counts_layer_integer_like",
            "value":
                True,
        },
        {
            "metric":
                "counts_encoding",
            "value":
                counts_encoding,
        },
        {
            "metric":
                "counts_dtype",
            "value":
                counts_dtype,
        },
        {
            "metric":
                "counts_nnz",
            "value":
                counts_nnz,
        },
        {
            "metric":
                "counts_sample_min",
            "value":
                float(
                    count_sample.min()
                ),
        },
        {
            "metric":
                "counts_sample_max",
            "value":
                float(
                    count_sample.max()
                ),
        },
        {
            "metric":
                "X_umap_present",
            "value":
                x_umap is not None,
        },
        {
            "metric":
                "X_scVI_present",
            "value":
                x_scvi_shape is not None,
        },
    ]
)


write_tsv_replace(
    readiness,
    DE_READINESS_TSV,
)

display(
    readiness
)

print(
    "\nCanonical cluster sizes:"
)

display(
    cluster_counts
    .sort_index(
        key=lambda index: [
            natural_cluster_key(x)
            for x in index
        ]
    )
    .to_frame(
        "n_cells"
    )
)

counts encoding: csr_matrix
counts shape: (351427, 15176)
counts nnz: 739913124
counts dtype: int32
obs rows: 351427
var rows: 15176
X_scVI shape: (351427, 30)
X_umap shape: (351427, 2)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_readiness_report.tsv shape=(14, 2)


,metric,value
0,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...
1,n_cells,351427
2,n_genes,15176
3,main_cluster_key,leiden_scvi_main
4,n_clusters,18
5,counts_layer_present,True
6,counts_layer_integer_like,True
7,counts_encoding,csr_matrix
8,counts_dtype,int32
9,counts_nnz,739913124



Canonical cluster sizes:


,n_cells
leiden_scvi_main,
0,61646
1,59791
2,38634
3,35355
4,32896
5,23977
6,22943
7,22525
8,14809


In [3]:
# =========================
# 2) CLUSTER SIZE / COMPOSITION QC
# =========================

cluster_size = (
    obs[
        MAIN_CLUSTER_KEY
    ]
    .value_counts()
    .rename_axis(
        "cluster"
    )
    .reset_index(
        name="n_cells"
    )
)

cluster_size[
    "cluster"
] = (
    cluster_size[
        "cluster"
    ]
    .astype(str)
)

cluster_size = (
    cluster_size
    .sort_values(
        "cluster",
        key=lambda series: series.map(
            natural_cluster_key
        ),
    )
)


def dominant_summary(
    obs_frame,
    cluster_key,
    column,
):
    tab = pd.crosstab(
        obs_frame[
            cluster_key
        ].astype(str),
        obs_frame[
            column
        ].astype(str),
    )

    frac = tab.div(
        tab.sum(
            axis=1
        ),
        axis=0,
    )

    return pd.DataFrame(
        {
            "cluster":
                tab.index.astype(str),

            f"top_{column}":
                frac.idxmax(
                    axis=1
                ).values,

            f"top_{column}_fraction":
                frac.max(
                    axis=1
                ).values,

            f"n_{column}":
                (
                    tab > 0
                ).sum(
                    axis=1
                ).values,
        }
    )


qc = cluster_size.copy()

for column in (
    DATASET_COL,
    CONDITION_COL,
    PATIENT_COL,
    SAMPLE_COL,
):
    qc = qc.merge(
        dominant_summary(
            obs,
            MAIN_CLUSTER_KEY,
            column,
        ),
        on="cluster",
        how="left",
    )


qc[
    "fraction_total_cells"
] = (
    qc[
        "n_cells"
    ]
    / EXPECTED_N_CELLS
)

qc[
    "small_cluster_lt_500"
] = (
    qc[
        "n_cells"
    ]
    < 500
)

qc[
    "small_cluster_lt_100"
] = (
    qc[
        "n_cells"
    ]
    < 100
)

qc[
    "dataset_dominant_gt_0_80"
] = (
    qc[
        f"top_{DATASET_COL}_fraction"
    ]
    > 0.80
)

qc[
    "condition_dominant_gt_0_80"
] = (
    qc[
        f"top_{CONDITION_COL}_fraction"
    ]
    > 0.80
)

qc[
    "patient_dominant_gt_0_80"
] = (
    qc[
        f"top_{PATIENT_COL}_fraction"
    ]
    > 0.80
)

qc[
    "sample_dominant_gt_0_80"
] = (
    qc[
        f"top_{SAMPLE_COL}_fraction"
    ]
    > 0.80
)


write_tsv_replace(
    qc,
    CLUSTER_QC_TSV,
)

display(
    qc
)


composition_rows = []

for column in (
    DATASET_COL,
    CONDITION_COL,
    PATIENT_COL,
    SAMPLE_COL,
):
    tab = pd.crosstab(
        obs[
            MAIN_CLUSTER_KEY
        ].astype(str),
        obs[
            column
        ].astype(str),
    )

    frac = tab.div(
        tab.sum(
            axis=1
        ),
        axis=0,
    )

    for cluster in tab.index.astype(str):
        for value in tab.columns.astype(str):
            composition_rows.append(
                {
                    "cluster":
                        cluster,
                    "metadata_column":
                        column,
                    "metadata_value":
                        value,
                    "n_cells":
                        int(
                            tab.loc[
                                cluster,
                                value,
                            ]
                        ),
                    "fraction_within_cluster":
                        float(
                            frac.loc[
                                cluster,
                                value,
                            ]
                        ),
                }
            )


composition = pd.DataFrame(
    composition_rows
)

write_tsv_replace(
    composition,
    CLUSTER_COMPOSITION_TSV,
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_cluster_QC_summary.tsv shape=(18, 21)


,cluster,n_cells,top_core_dataset,top_core_dataset_fraction,n_core_dataset,top_condition,top_condition_fraction,n_condition,top_patient_id,top_patient_id_fraction,...,top_sample_id,top_sample_id_fraction,n_sample_id,fraction_total_cells,small_cluster_lt_500,small_cluster_lt_100,dataset_dominant_gt_0_80,condition_dominant_gt_0_80,patient_dominant_gt_0_80,sample_dominant_gt_0_80
0,0,61646,DS3_GSE182109,0.555900,4,GBM,0.444100,4,04,0.185527,...,GSM5518602_ndGBM-01-D,0.064692,79,0.175416,False,False,False,False,False,False
1,1,59791,DS3_GSE182109,0.669649,4,ndGBM,0.426586,4,01,0.148534,...,GSM5518603_ndGBM-01-F,0.062953,82,0.170138,False,False,False,False,False,False
2,2,38634,DS3_GSE182109,0.552674,4,GBM,0.447326,4,04,0.240358,...,GSM5518621_rGBM-04-3,0.057203,75,0.109935,False,False,False,False,False,False
3,3,35355,DS3_GSE182109,0.746599,4,rGBM,0.586225,4,01,0.470117,...,GSM5518597_rGBM-01-B,0.217282,79,0.100604,False,False,False,False,False,False
4,4,32896,DS4_GSE278450,0.466652,4,GBM,0.606548,4,01,0.151751,...,GSM8546833_BT356,0.149471,75,0.093607,False,False,False,False,False,False
5,5,23977,DS3_GSE182109,0.967052,4,rGBM,0.617091,4,02,0.555824,...,GSM5518615_rGBM-02-5,0.194603,78,0.068228,False,False,True,False,False,False
6,6,22943,DS3_GSE182109,0.633483,4,GBM,0.366517,4,02,0.157259,...,GSM5518636_ndGBM-07,0.089134,77,0.065285,False,False,False,False,False,False
7,7,22525,DS3_GSE182109,0.385838,4,GBM,0.614162,4,04,0.121154,...,GSM8546834_BT363,0.120844,80,0.064096,False,False,False,False,False,False
8,8,14809,DS4_GSE278450,0.558714,4,GBM,0.610777,4,04,0.179485,...,GSM8546845_BT396,0.113445,72,0.042140,False,False,False,False,False,False
9,9,13864,DS3_GSE182109,0.941864,4,ndGBM,0.806621,4,10,0.696336,...,GSM5518629_ndGBM-10,0.696336,78,0.039451,False,False,True,True,False,False


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_cluster_composition_summary.tsv shape=(2538, 5)


In [4]:
# =========================
# 3) MARKER PANEL PRESENCE
# =========================

marker_panels = {
    "Tumor_AC_like": [
        "GFAP", "AQP4", "ALDH1L1",
        "SLC1A3", "CLU", "APOE",
    ],
    "Tumor_OPC_like": [
        "PDGFRA", "OLIG1", "OLIG2",
        "SOX10", "CSPG4", "BCAN",
    ],
    "Tumor_NPC_like": [
        "SOX2", "NES", "VIM",
        "FABP7", "DLL3", "ASCL1",
    ],
    "Tumor_MES_like": [
        "CD44", "CHI3L1", "SERPINE1",
        "VIM", "TNC", "EMP3",
    ],
    "Cycling": [
        "MKI67", "TOP2A", "UBE2C",
        "HMGB2", "STMN1", "CENPF",
    ],
    "Hypoxia_stress": [
        "VEGFA", "CA9", "DDIT4",
        "HILPDA", "ENO1", "HMOX1",
    ],
    "Myeloid": [
        "PTPRC", "TYROBP", "LST1",
        "C1QA", "C1QB", "C1QC",
        "CSF1R", "AIF1",
    ],
    "Inflammatory_myeloid": [
        "IL1B", "CCL3", "CCL4",
        "CXCL8", "NFKBIA",
    ],
    "T_NK": [
        "CD3D", "CD3E", "TRAC",
        "IL7R", "NKG7", "GNLY",
        "GZMB",
    ],
    "B_Plasma": [
        "MS4A1", "CD79A", "MZB1",
        "JCHAIN", "SDC1",
    ],
    "Oligodendrocyte": [
        "MBP", "PLP1", "MOG",
        "MAG", "MOBP",
    ],
    "Endothelial": [
        "PECAM1", "VWF", "KDR",
        "FLT1", "CLDN5",
    ],
    "Mural_Pericyte": [
        "PDGFRB", "RGS5", "ACTA2",
        "TAGLN", "MCAM",
    ],
    "Fibroblast_ECM": [
        "COL1A1", "COL1A2", "DCN",
        "LUM", "COL3A1",
    ],
}


var_names = pd.Index(
    var.index.astype(str)
)

var_set = set(
    var_names
)

presence_rows = []

for panel, genes in marker_panels.items():
    for gene in genes:
        presence_rows.append(
            {
                "panel":
                    panel,
                "gene":
                    gene,
                "present_in_fullgene_object":
                    gene in var_set,
            }
        )


marker_presence = pd.DataFrame(
    presence_rows
)

write_tsv_replace(
    marker_presence,
    MARKER_PANEL_PRESENCE_TSV,
)

presence_summary = (
    marker_presence
    .groupby(
        "panel"
    )[
        "present_in_fullgene_object"
    ]
    .agg(
        [
            "sum",
            "count",
        ]
    )
)

display(
    presence_summary
)


missing_marker_panel_genes = (
    marker_presence.loc[
        ~marker_presence[
            "present_in_fullgene_object"
        ],
        [
            "panel",
            "gene",
        ],
    ]
    .rename(
        columns={
            "gene":
                "missing_gene"
        }
    )
    .reset_index(
        drop=True
    )
)

write_tsv_replace(
    missing_marker_panel_genes,
    MISSING_MARKER_PANEL_TSV,
)


expected_missing = {
    "MS4A1",
    "JCHAIN",
}

observed_missing = set(
    missing_marker_panel_genes[
        "missing_gene"
    ].astype(str)
)

if observed_missing != expected_missing:
    raise AssertionError(
        "Marker-panel presence no longer matches "
        "the archived execution. "
        f"Observed missing genes: {sorted(observed_missing)}"
    )


print(
    "Historical marker-panel presence reproduced:",
    f"{int(marker_presence['present_in_fullgene_object'].sum())}/"
    f"{len(marker_presence)}",
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_marker_panel_presence.tsv shape=(81, 3)


,sum,count
panel,,
B_Plasma,3,5
Cycling,6,6
Endothelial,5,5
Fibroblast_ECM,5,5
Hypoxia_stress,6,6
Inflammatory_myeloid,5,5
Mural_Pericyte,5,5
Myeloid,8,8
Oligodendrocyte,5,5


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_missing_marker_panel_genes.tsv shape=(2, 2)
Historical marker-panel presence reproduced: 79/81


In [5]:
# =========================
# 4) PREPARE / RESUME HISTORICAL LOG1P X + NVMe GENE-BLOCK CACHE
# =========================

n_blocks = math.ceil(
    EXPECTED_N_GENES
    / GENE_BLOCK_SIZE
)


def block_path(
    block_id,
):
    start = (
        block_id
        * GENE_BLOCK_SIZE
    )
    end = min(
        start
        + GENE_BLOCK_SIZE,
        EXPECTED_N_GENES,
    )

    return (
        BLOCK_CACHE_DIR
        / (
            f"block_{block_id:03d}_"
            f"g{start:05d}_{end:05d}.bin"
        )
    )


def source_fingerprint_matches(
    payload,
):
    return (
        payload.get(
            "input_fingerprint"
        )
        == input_fingerprint
        and payload.get(
            "gene_block_size"
        )
        == GENE_BLOCK_SIZE
        and payload.get(
            "record_dtype"
        )
        == BLOCK_RECORD_DTYPE.descr
    )


def load_normalization_progress():
    if not NORMALIZATION_PROGRESS_JSON.exists():
        return None

    try:
        with NORMALIZATION_PROGRESS_JSON.open(
            "r",
            encoding="utf-8",
        ) as handle:
            payload = json.load(
                handle
            )
    except Exception:
        return None

    if not source_fingerprint_matches(
        payload
    ):
        return None

    return payload


def truncate_block_files(
    committed_bytes,
):
    for block_id in range(
        n_blocks
    ):
        path = block_path(
            block_id
        )

        target_bytes = int(
            committed_bytes[
                block_id
            ]
        )

        if not path.exists():
            if target_bytes != 0:
                raise RuntimeError(
                    "Committed cache block is missing: "
                    f"{path}"
                )
            continue

        with path.open(
            "r+b"
        ) as handle:
            handle.truncate(
                target_bytes
            )


progress = load_normalization_progress()


if (
    progress is None
    or not MARKER_READY_BUILDING_H5AD.exists()
):
    print(
        "Initializing fresh memory-safe marker-DE build."
    )

    if MARKER_READY_BUILDING_H5AD.exists():
        MARKER_READY_BUILDING_H5AD.unlink()

    for path in BLOCK_CACHE_DIR.glob(
        "block_*.bin"
    ):
        path.unlink()

    if NORMALIZATION_PROGRESS_JSON.exists():
        NORMALIZATION_PROGRESS_JSON.unlink()


    input_size = int(
        FULLGENE_INPUT_H5AD.stat().st_size
    )

    free_space = int(
        shutil.disk_usage(
            MARKER_READY_BUILDING_H5AD.parent
        ).free
    )

    # The copied backbone plus a new float32 normalized X data vector
    # can transiently exceed the input file size.
    minimum_required = (
        input_size
        + counts_nnz * 4
        + 5 * 1024**3
    )

    print(
        "Input GiB:",
        round(
            input_size
            / 1024**3,
            2,
        ),
    )

    print(
        "Free GiB:",
        round(
            free_space
            / 1024**3,
            2,
        ),
    )

    if free_space < minimum_required:
        raise OSError(
            "Not enough free space for the "
            "memory-safe marker-ready H5AD build."
        )


    print(
        "[COPY] canonical full-gene backbone"
    )

    shutil.copyfile(
        FULLGENE_INPUT_H5AD,
        MARKER_READY_BUILDING_H5AD,
    )

    print(
        "[COPY COMPLETE]"
    )


    with h5py.File(
        MARKER_READY_BUILDING_H5AD,
        "r+",
    ) as handle:

        counts = (
            handle[
                "layers"
            ][
                "counts"
            ]
        )

        if "X" in handle:
            del handle[
                "X"
            ]

        x_group = handle.create_group(
            "X"
        )

        x_group.attrs[
            "encoding-type"
        ] = "csr_matrix"

        x_group.attrs[
            "encoding-version"
        ] = "0.1.0"

        x_group.attrs[
            "shape"
        ] = np.asarray(
            [
                EXPECTED_N_CELLS,
                EXPECTED_N_GENES,
            ],
            dtype=np.int64,
        )

        # Share CSR structure with layers['counts'] inside this HDF5 file.
        x_group[
            "indices"
        ] = counts[
            "indices"
        ]

        x_group[
            "indptr"
        ] = counts[
            "indptr"
        ]

        data_chunk = min(
            counts_nnz,
            1_000_000,
        )

        x_group.create_dataset(
            "data",
            shape=(
                counts_nnz,
            ),
            dtype=np.float32,
            chunks=(
                data_chunk,
            ),
            compression="lzf",
        )

        handle.flush()


    progress = {
        "input_fingerprint":
            input_fingerprint,

        "gene_block_size":
            GENE_BLOCK_SIZE,

        "record_dtype":
            BLOCK_RECORD_DTYPE.descr,

        "next_row":
            0,

        "normalization_complete":
            False,

        "committed_block_bytes":
            [
                0
                for _
                in range(
                    n_blocks
                )
            ],
    }

    write_json_atomic(
        progress,
        NORMALIZATION_PROGRESS_JSON,
    )


# Remove any uncommitted appended records left by an interrupted row block.
truncate_block_files(
    progress[
        "committed_block_bytes"
    ]
)


if progress[
    "normalization_complete"
]:
    print(
        "[REUSING] completed normalized X and gene-block cache."
    )

else:
    next_row = int(
        progress[
            "next_row"
        ]
    )

    print(
        "Resuming normalization/cache at row:",
        next_row,
    )


    with h5py.File(
        MARKER_READY_BUILDING_H5AD,
        "r+",
    ) as handle:

        counts = (
            handle[
                "layers"
            ][
                "counts"
            ]
        )

        raw_data_ds = counts[
            "data"
        ]

        raw_indices_ds = counts[
            "indices"
        ]

        indptr = np.asarray(
            counts[
                "indptr"
            ],
            dtype=np.int64,
        )

        normalized_data_ds = (
            handle[
                "X"
            ][
                "data"
            ]
        )


        r0 = next_row

        while r0 < EXPECTED_N_CELLS:

            d0 = int(
                indptr[
                    r0
                ]
            )

            target_d1 = min(
                d0
                + MAX_NNZ_PER_ROW_BLOCK,
                counts_nnz,
            )

            r1 = int(
                np.searchsorted(
                    indptr,
                    target_d1,
                    side="right",
                )
                - 1
            )

            r1 = max(
                r1,
                r0 + 1,
            )

            r1 = min(
                r1,
                EXPECTED_N_CELLS,
            )

            d1 = int(
                indptr[
                    r1
                ]
            )


            raw_values = np.asarray(
                raw_data_ds[
                    d0:d1
                ],
                dtype=np.float32,
            )

            columns = np.asarray(
                raw_indices_ds[
                    d0:d1
                ],
                dtype=np.int32,
            )

            row_nnz = np.diff(
                indptr[
                    r0:r1 + 1
                ]
            ).astype(
                np.int64,
                copy=False,
            )

            rows = np.repeat(
                np.arange(
                    r0,
                    r1,
                    dtype=np.int32,
                ),
                row_nnz,
            )


            # Historical Scanpy semantics:
            # float32 counts -> counts_per_cell / 1e4 ->
            # X /= norm_factor -> log1p.
            row_sums = np.bincount(
                (
                    rows
                    - np.int32(
                        r0
                    )
                ).astype(
                    np.int64,
                    copy=False,
                ),
                weights=raw_values,
                minlength=(
                    r1
                    - r0
                ),
            ).astype(
                np.float32
            )

            if np.any(
                row_sums <= 0
            ):
                raise ValueError(
                    "Encountered zero-count cell during normalization."
                )

            norm_factor = (
                row_sums
                / TARGET_SUM
            ).astype(
                np.float32,
                copy=False,
            )

            normalized_values = (
                raw_values
                / norm_factor[
                    rows
                    - np.int32(
                        r0
                    )
                ]
            ).astype(
                np.float32,
                copy=False,
            )

            np.log1p(
                normalized_values,
                out=normalized_values,
            )


            # Write normalized X in original CSR nnz order.
            normalized_data_ds[
                d0:d1
            ] = normalized_values


            # Partition the same normalized entries into NVMe gene blocks.
            block_ids = (
                columns
                // GENE_BLOCK_SIZE
            ).astype(
                np.uint8,
                copy=False,
            )

            local_columns = (
                columns
                - (
                    block_ids.astype(
                        np.int32
                    )
                    * GENE_BLOCK_SIZE
                )
            ).astype(
                np.uint16,
                copy=False,
            )


            order = np.argsort(
                block_ids,
                kind="stable",
            )

            sorted_block_ids = (
                block_ids[
                    order
                ]
            )

            sorted_rows = (
                rows[
                    order
                ]
            )

            sorted_columns = (
                local_columns[
                    order
                ]
            )

            sorted_values = (
                normalized_values[
                    order
                ]
            )


            block_counts = np.bincount(
                sorted_block_ids,
                minlength=n_blocks,
            ).astype(
                np.int64,
                copy=False,
            )

            boundaries = np.concatenate(
                [
                    np.array(
                        [0],
                        dtype=np.int64,
                    ),
                    np.cumsum(
                        block_counts,
                        dtype=np.int64,
                    ),
                ]
            )


            for block_id in np.flatnonzero(
                block_counts
            ):
                left = int(
                    boundaries[
                        block_id
                    ]
                )

                right = int(
                    boundaries[
                        block_id
                        + 1
                    ]
                )

                records = np.empty(
                    right - left,
                    dtype=BLOCK_RECORD_DTYPE,
                )

                records[
                    "row"
                ] = sorted_rows[
                    left:right
                ]

                records[
                    "col"
                ] = sorted_columns[
                    left:right
                ]

                records[
                    "val"
                ] = sorted_values[
                    left:right
                ]

                with block_path(
                    int(
                        block_id
                    )
                ).open(
                    "ab"
                ) as block_handle:
                    records.tofile(
                        block_handle
                    )


            handle.flush()


            committed_bytes = []

            for block_id in range(
                n_blocks
            ):
                path = block_path(
                    block_id
                )

                committed_bytes.append(
                    int(
                        path.stat().st_size
                    )
                    if path.exists()
                    else 0
                )


            progress[
                "next_row"
            ] = int(
                r1
            )

            progress[
                "committed_block_bytes"
            ] = committed_bytes

            write_json_atomic(
                progress,
                NORMALIZATION_PROGRESS_JSON,
            )


            print(
                "[NORMALIZED+CACHED]",
                f"rows {r0:,}-{r1:,}",
                f"nnz {d0:,}-{d1:,}",
            )


            del raw_values
            del columns
            del row_nnz
            del rows
            del row_sums
            del norm_factor
            del normalized_values
            del block_ids
            del local_columns
            del order
            del sorted_block_ids
            del sorted_rows
            del sorted_columns
            del sorted_values

            gc.collect()

            r0 = r1


        progress[
            "normalization_complete"
        ] = True

        write_json_atomic(
            progress,
            NORMALIZATION_PROGRESS_JSON,
        )


# Add the same preprocessing provenance used by the historical notebook.
with h5py.File(
    MARKER_READY_BUILDING_H5AD,
    "r+",
) as handle:

    if "uns" not in handle:
        write_elem(
            handle,
            "uns",
            {},
        )

    preprocessing_info = {
        "cluster_key":
            MAIN_CLUSTER_KEY,

        "counts_source":
            "adata.layers['counts']",

        "normalization":
            "scanpy.pp.normalize_total(target_sum=1e4)",

        "transformation":
            "scanpy.pp.log1p",

        "rank_method":
            (
                "scanpy.tl.rank_genes_groups("
                "method='wilcoxon', tie_correct=True, pts=True)"
            ),

        "use_raw":
            False,

        "memory_safe_implementation":
            (
                "row-sequential normalization plus "
                "gene-block Scanpy Wilcoxon"
            ),

        "note":
            (
                "Scientific DE semantics match the archived "
                "whole-matrix analysis; only execution is blocked."
            ),
    }

    if (
        "marker_DE_preprocessing_HVG8000_r0_6"
        in handle[
            "uns"
        ]
    ):
        del handle[
            "uns"
        ][
            "marker_DE_preprocessing_HVG8000_r0_6"
        ]

    write_elem(
        handle[
            "uns"
        ],
        "marker_DE_preprocessing_HVG8000_r0_6",
        preprocessing_info,
    )

    # Historical sc.pp.log1p records this.
    if "log1p" in handle[
        "uns"
    ]:
        del handle[
            "uns"
        ][
            "log1p"
        ]

    write_elem(
        handle[
            "uns"
        ],
        "log1p",
        {
            "base":
                None
        },
    )

    handle.flush()


print(
    "\n[NORMALIZATION CACHE READY]"
)

print(
    MARKER_READY_BUILDING_H5AD
)

print(
    "Gene blocks:",
    n_blocks,
)

Initializing fresh memory-safe marker-DE build.
Input GiB: 11.5
Free GiB: 762.69
[COPY] canonical full-gene backbone
[COPY COMPLETE]
Resuming normalization/cache at row: 0
[NORMALIZED+CACHED] rows 0-4,241 nnz 0-7,998,673
[NORMALIZED+CACHED] rows 4,241-11,672 nnz 7,998,673-15,998,368
[NORMALIZED+CACHED] rows 11,672-21,842 nnz 15,998,368-23,998,353
[NORMALIZED+CACHED] rows 21,842-31,237 nnz 23,998,353-31,998,154
[NORMALIZED+CACHED] rows 31,237-39,049 nnz 31,998,154-39,996,920
[NORMALIZED+CACHED] rows 39,049-45,782 nnz 39,996,920-47,995,616
[NORMALIZED+CACHED] rows 45,782-49,725 nnz 47,995,616-55,994,798
[NORMALIZED+CACHED] rows 49,725-54,177 nnz 55,994,798-63,992,114
[NORMALIZED+CACHED] rows 54,177-58,699 nnz 63,992,114-71,991,266
[NORMALIZED+CACHED] rows 58,699-63,156 nnz 71,991,266-79,990,512
[NORMALIZED+CACHED] rows 63,156-67,223 nnz 79,990,512-87,990,258
[NORMALIZED+CACHED] rows 67,223-70,841 nnz 87,990,258-95,987,949
[NORMALIZED+CACHED] rows 70,841-74,149 nnz 95,987,949-103,986,492


In [6]:
# =========================
# 5) INITIALIZE / RESUME BLOCKWISE SCANPY WILCOXON
# =========================

cluster_labels = (
    obs[
        MAIN_CLUSTER_KEY
    ]
    .astype(str)
    .astype(
        "category"
    )
)

cluster_order = [
    str(x)
    for x
    in cluster_labels.cat.categories
]

n_groups = len(
    cluster_order
)

if n_groups != EXPECTED_N_CLUSTERS:
    raise AssertionError(
        "Unexpected number of cluster categories."
    )


scores_path = (
    DE_STATE_DIR
    / "scores.npy"
)

pvals_path = (
    DE_STATE_DIR
    / "pvals.npy"
)

logfc_path = (
    DE_STATE_DIR
    / "logfoldchanges.npy"
)

pts_path = (
    DE_STATE_DIR
    / "pts.npy"
)

pts_rest_path = (
    DE_STATE_DIR
    / "pts_rest.npy"
)


def de_progress_valid(
    payload,
):
    return (
        payload.get(
            "input_fingerprint"
        )
        == input_fingerprint
        and payload.get(
            "gene_block_size"
        )
        == GENE_BLOCK_SIZE
        and payload.get(
            "cluster_order"
        )
        == cluster_order
        and payload.get(
            "normalization_complete"
        )
        is True
    )


de_progress = None

if DE_PROGRESS_JSON.exists():
    try:
        with DE_PROGRESS_JSON.open(
            "r",
            encoding="utf-8",
        ) as handle:
            candidate_progress = json.load(
                handle
            )

        if de_progress_valid(
            candidate_progress
        ):
            de_progress = candidate_progress

    except Exception:
        de_progress = None


state_paths = [
    scores_path,
    pvals_path,
    logfc_path,
    pts_path,
    pts_rest_path,
]


if (
    de_progress is None
    or not all(
        path.exists()
        for path in state_paths
    )
):
    print(
        "Initializing fresh blockwise DE state."
    )

    for path in state_paths:
        if path.exists():
            path.unlink()


    scores_mm = (
        np.lib.format.open_memmap(
            scores_path,
            mode="w+",
            dtype=np.float32,
            shape=(
                n_groups,
                EXPECTED_N_GENES,
            ),
        )
    )

    pvals_mm = (
        np.lib.format.open_memmap(
            pvals_path,
            mode="w+",
            dtype=np.float64,
            shape=(
                n_groups,
                EXPECTED_N_GENES,
            ),
        )
    )

    logfc_mm = (
        np.lib.format.open_memmap(
            logfc_path,
            mode="w+",
            dtype=np.float32,
            shape=(
                n_groups,
                EXPECTED_N_GENES,
            ),
        )
    )

    pts_mm = (
        np.lib.format.open_memmap(
            pts_path,
            mode="w+",
            dtype=np.float64,
            shape=(
                n_groups,
                EXPECTED_N_GENES,
            ),
        )
    )

    pts_rest_mm = (
        np.lib.format.open_memmap(
            pts_rest_path,
            mode="w+",
            dtype=np.float64,
            shape=(
                n_groups,
                EXPECTED_N_GENES,
            ),
        )
    )


    for array in (
        scores_mm,
        pvals_mm,
        logfc_mm,
        pts_mm,
        pts_rest_mm,
    ):
        array[:] = np.nan
        array.flush()


    de_progress = {
        "input_fingerprint":
            input_fingerprint,

        "gene_block_size":
            GENE_BLOCK_SIZE,

        "cluster_order":
            cluster_order,

        "normalization_complete":
            True,

        "completed_blocks":
            [],
    }

    write_json_atomic(
        de_progress,
        DE_PROGRESS_JSON,
    )

else:
    scores_mm = np.load(
        scores_path,
        mmap_mode="r+",
    )

    pvals_mm = np.load(
        pvals_path,
        mmap_mode="r+",
    )

    logfc_mm = np.load(
        logfc_path,
        mmap_mode="r+",
    )

    pts_mm = np.load(
        pts_path,
        mmap_mode="r+",
    )

    pts_rest_mm = np.load(
        pts_rest_path,
        mmap_mode="r+",
    )

    print(
        "[REUSING] partial DE state."
    )


completed_blocks = set(
    int(x)
    for x
    in de_progress[
        "completed_blocks"
    ]
)


block_obs = pd.DataFrame(
    {
        MAIN_CLUSTER_KEY:
            cluster_labels,
    },
    index=obs.index.copy(),
)


def load_block_csr(
    block_id,
):
    start = (
        block_id
        * GENE_BLOCK_SIZE
    )

    end = min(
        start
        + GENE_BLOCK_SIZE,
        EXPECTED_N_GENES,
    )

    block_gene_count = (
        end
        - start
    )

    path = block_path(
        block_id
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    record_bytes = int(
        path.stat().st_size
    )

    if (
        record_bytes
        % BLOCK_RECORD_DTYPE.itemsize
        != 0
    ):
        raise RuntimeError(
            "Corrupt gene-block cache file: "
            f"{path}"
        )


    records = np.memmap(
        path,
        dtype=BLOCK_RECORD_DTYPE,
        mode="r",
    )

    # IMPORTANT:
    # np.memmap fields are read-only views here. Scanpy calls
    # scipy.sparse.csr_matrix.eliminate_zeros() in-place during
    # rank_genes_groups, so CSR data/indices must be writable.
    # Materialize only this small gene block as writable NumPy arrays.
    rows = np.array(
        records[
            "row"
        ],
        dtype=np.int32,
        copy=True,
    )

    local_columns = np.array(
        records[
            "col"
        ],
        dtype=np.int32,
        copy=True,
    )

    values = np.array(
        records[
            "val"
        ],
        dtype=np.float32,
        copy=True,
    )


    if len(rows):
        if (
            rows.min() < 0
            or rows.max()
            >= EXPECTED_N_CELLS
        ):
            raise RuntimeError(
                "Cached row index is out of range."
            )

        if (
            local_columns.min() < 0
            or local_columns.max()
            >= block_gene_count
        ):
            raise RuntimeError(
                "Cached local gene index is out of range."
            )


    # Records are appended in ascending row-block order and stable CSR order.
    if (
        len(rows) > 1
        and np.any(
            rows[
                1:
            ]
            < rows[
                :-1
            ]
        )
    ):
        raise RuntimeError(
            "Gene-block cache is not row ordered."
        )


    row_counts = np.bincount(
        rows.astype(
            np.int64,
            copy=False,
        ),
        minlength=EXPECTED_N_CELLS,
    ).astype(
        np.int64,
        copy=False,
    )

    indptr = np.empty(
        EXPECTED_N_CELLS + 1,
        dtype=np.int64,
    )

    indptr[
        0
    ] = 0

    np.cumsum(
        row_counts,
        out=indptr[
            1:
        ],
    )


    matrix = sp.csr_matrix(
        (
            values,
            local_columns,
            indptr,
        ),
        shape=(
            EXPECTED_N_CELLS,
            block_gene_count,
        ),
        dtype=np.float32,
        copy=False,
    )

    # Defensive check for SciPy/NumPy combinations that may propagate
    # non-writable flags into sparse internals.
    if (
        not matrix.data.flags.writeable
        or not matrix.indices.flags.writeable
        or not matrix.indptr.flags.writeable
    ):
        matrix = matrix.copy()


    return (
        matrix,
        start,
        end,
        records,
    )


TMP_RANK_KEY = (
    "_tmp_block_rank_genes_groups"
)


for block_id in range(
    n_blocks
):
    if block_id in completed_blocks:
        print(
            "[SKIP DE BLOCK]",
            block_id,
        )
        continue


    block_matrix, start, end, records = (
        load_block_csr(
            block_id
        )
    )

    block_genes = (
        var_names[
            start:end
        ]
        .astype(str)
    )


    block_adata = ad.AnnData(
        X=block_matrix,
        obs=block_obs.copy(),
        var=pd.DataFrame(
            index=block_genes.copy(),
        ),
    )

    block_adata.uns[
        "log1p"
    ] = {
        "base":
            None
    }


    print(
        "\n[DE BLOCK]",
        block_id,
        f"genes {start:,}-{end:,}",
        f"nnz={block_matrix.nnz:,}",
    )


    sc.tl.rank_genes_groups(
        block_adata,
        groupby=MAIN_CLUSTER_KEY,
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        n_genes=block_adata.n_vars,
        use_raw=False,
        pts=True,
        tie_correct=True,
        rankby_abs=False,
        key_added=TMP_RANK_KEY,
    )


    result = (
        block_adata.uns[
            TMP_RANK_KEY
        ]
    )

    names_df = pd.DataFrame(
        result[
            "names"
        ]
    )

    scores_df = pd.DataFrame(
        result[
            "scores"
        ]
    )

    pvals_df = pd.DataFrame(
        result[
            "pvals"
        ]
    )

    logfc_df = pd.DataFrame(
        result[
            "logfoldchanges"
        ]
    )


    for group_index, group in enumerate(
        cluster_order
    ):
        names = (
            names_df[
                group
            ]
            .astype(str)
            .to_numpy()
        )

        local_positions = (
            pd.Index(
                block_genes
            )
            .get_indexer(
                names
            )
        )

        if np.any(
            local_positions < 0
        ):
            raise RuntimeError(
                "Could not map block DE gene names "
                "back to the block var order."
            )

        global_positions = (
            start
            + local_positions
        )


        scores_mm[
            group_index,
            global_positions,
        ] = (
            scores_df[
                group
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        pvals_mm[
            group_index,
            global_positions,
        ] = (
            pvals_df[
                group
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        logfc_mm[
            group_index,
            global_positions,
        ] = (
            logfc_df[
                group
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        pts_mm[
            group_index,
            start:end,
        ] = (
            result[
                "pts"
            ][
                group
            ]
            .reindex(
                block_genes
            )
            .to_numpy(
                dtype=np.float64
            )
        )

        pts_rest_mm[
            group_index,
            start:end,
        ] = (
            result[
                "pts_rest"
            ][
                group
            ]
            .reindex(
                block_genes
            )
            .to_numpy(
                dtype=np.float64
            )
        )


    for array in (
        scores_mm,
        pvals_mm,
        logfc_mm,
        pts_mm,
        pts_rest_mm,
    ):
        array.flush()


    completed_blocks.add(
        block_id
    )

    de_progress[
        "completed_blocks"
    ] = sorted(
        completed_blocks
    )

    write_json_atomic(
        de_progress,
        DE_PROGRESS_JSON,
    )


    del block_adata
    del block_matrix
    del records
    del result
    del names_df
    del scores_df
    del pvals_df
    del logfc_df

    gc.collect()


print(
    "\nALL DE BLOCKS COMPLETE:",
    len(
        completed_blocks
    ),
    "/",
    n_blocks,
)

[REUSING] partial DE state.

[DE BLOCK] 0 genes 0-256 nnz=9,728,287
ranking genes
    finished (0:00:53)

[DE BLOCK] 1 genes 256-512 nnz=10,275,639
ranking genes
    finished (0:00:30)

[DE BLOCK] 2 genes 512-768 nnz=12,871,349
ranking genes
    finished (0:00:33)

[DE BLOCK] 3 genes 768-1,024 nnz=12,962,198
ranking genes
    finished (0:00:28)

[DE BLOCK] 4 genes 1,024-1,280 nnz=12,885,885
ranking genes
    finished (0:00:28)

[DE BLOCK] 5 genes 1,280-1,536 nnz=12,904,113
ranking genes
    finished (0:00:26)

[DE BLOCK] 6 genes 1,536-1,792 nnz=10,717,984
ranking genes
    finished (0:00:25)

[DE BLOCK] 7 genes 1,792-2,048 nnz=12,269,112
ranking genes
    finished (0:00:28)

[DE BLOCK] 8 genes 2,048-2,304 nnz=10,993,251
ranking genes
    finished (0:00:26)

[DE BLOCK] 9 genes 2,304-2,560 nnz=13,067,049
ranking genes
    finished (0:00:26)

[DE BLOCK] 10 genes 2,560-2,816 nnz=14,285,869
ranking genes
    finished (0:00:27)

[DE BLOCK] 11 genes 2,816-3,072 nnz=13,511,021
ranking genes
  

In [7]:
# =========================
# 6) GLOBAL BH CORRECTION + RECONSTRUCT HISTORICAL SCANPY RANKING
# =========================

for array, name in (
    (
        scores_mm,
        "scores",
    ),
    (
        pvals_mm,
        "pvals",
    ),
    (
        logfc_mm,
        "logfoldchanges",
    ),
    (
        pts_mm,
        "pts",
    ),
    (
        pts_rest_mm,
        "pts_rest",
    ),
):
    if np.isnan(
        np.asarray(
            array
        )
    ).any():
        raise RuntimeError(
            f"Incomplete DE state: {name} contains NaN."
        )


pvals_adj = np.empty(
    (
        n_groups,
        EXPECTED_N_GENES,
    ),
    dtype=np.float64,
)


for group_index in range(
    n_groups
):
    pvals_group = np.asarray(
        pvals_mm[
            group_index
        ],
        dtype=np.float64,
    ).copy()

    pvals_group[
        np.isnan(
            pvals_group
        )
    ] = 1.0

    _, adjusted, _, _ = multipletests(
        pvals_group,
        alpha=0.05,
        method="fdr_bh",
    )

    pvals_adj[
        group_index
    ] = adjusted


def scanpy_select_top_n_all(
    scores,
):
    """
    Reproduce Scanpy's _select_top_n when n_top == n_genes.
    """
    n_from = scores.shape[
        0
    ]

    reference_indices = np.arange(
        n_from,
        dtype=int,
    )

    partition = np.argpartition(
        scores,
        -n_from,
    )[
        -n_from:
    ]

    partial_indices = np.argsort(
        scores[
            partition
        ]
    )[
        ::-1
    ]

    return reference_indices[
        partition[
            partial_indices
        ]
    ]


all_tables = []


for group_index, cluster in enumerate(
    cluster_order
):
    scores_group = np.asarray(
        scores_mm[
            group_index
        ],
        dtype=np.float32,
    )

    order = scanpy_select_top_n_all(
        scores_group
    )


    table = pd.DataFrame(
        {
            "cluster":
                cluster,

            "rank_scanpy":
                np.arange(
                    1,
                    EXPECTED_N_GENES + 1,
                    dtype=np.int32,
                ),

            "gene":
                var_names[
                    order
                ].astype(str),

            "score":
                scores_group[
                    order
                ].astype(
                    np.float32,
                    copy=False,
                ),

            "logfoldchange":
                np.asarray(
                    logfc_mm[
                        group_index
                    ],
                    dtype=np.float32,
                )[
                    order
                ],

            "pval":
                np.asarray(
                    pvals_mm[
                        group_index
                    ],
                    dtype=np.float64,
                )[
                    order
                ],

            "padj":
                pvals_adj[
                    group_index,
                    order,
                ],

            "pct_in_cluster":
                np.asarray(
                    pts_mm[
                        group_index
                    ],
                    dtype=np.float64,
                )[
                    order
                ],

            "pct_rest":
                np.asarray(
                    pts_rest_mm[
                        group_index
                    ],
                    dtype=np.float64,
                )[
                    order
                ],
        }
    )


    table[
        "pct_diff_in_minus_rest"
    ] = (
        table[
            "pct_in_cluster"
        ]
        - table[
            "pct_rest"
        ]
    )

    table[
        "direction"
    ] = np.where(
        table[
            "logfoldchange"
        ]
        >= 0,
        "up",
        "down",
    )

    all_tables.append(
        table
    )


all_de = pd.concat(
    all_tables,
    axis=0,
    ignore_index=True,
)


if all_de.shape != HISTORICAL_ALL_DE_SHAPE:
    raise AssertionError(
        "All-DE table shape no longer matches "
        f"the archived execution: {all_de.shape}"
    )


write_tsv_replace(
    all_de,
    DE_ALL_TSV,
)

display(
    all_de.head()
)

print(
    "All-DE shape:",
    all_de.shape,
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_all_genes.tsv shape=(273168, 11)


,cluster,rank_scanpy,gene,score,logfoldchange,pval,padj,pct_in_cluster,pct_rest,pct_diff_in_minus_rest,direction
0,0,1,PTPRZ1,307.749878,3.759407,0.0,0.0,0.941667,0.385167,0.556500,up
1,0,2,BCAN,288.052246,3.389220,0.0,0.0,0.845635,0.283411,0.562224,up
2,0,3,OLIG1,280.849701,3.234675,0.0,0.0,0.768760,0.231523,0.537237,up
3,0,4,NOVA1,271.248901,2.935542,0.0,0.0,0.856422,0.337372,0.519050,up
4,0,5,PTN,269.065308,3.289008,0.0,0.0,0.954368,0.466535,0.487833,up


All-DE shape: (273168, 11)


In [8]:
# =========================
# 7) HISTORICAL TOP / FILTERED TABLES + REPRODUCIBILITY CHECKPOINTS
# =========================

top_tables = []
top20_up_tables = []
filtered_tables = []


for cluster, sub in all_de.groupby(
    "cluster",
    sort=False,
):
    sub = sub.copy()


    up = sub[
        (
            sub[
                "logfoldchange"
            ]
            > 0
        )
        & (
            sub[
                "score"
            ]
            > 0
        )
    ].copy()

    up = up.sort_values(
        [
            "padj",
            "pval",
            "score",
            "logfoldchange",
        ],
        ascending=[
            True,
            True,
            False,
            False,
        ],
    )

    up = up.head(
        N_TOP_DIRECTION
    )

    up[
        "direction"
    ] = "up"

    up[
        "direction_rank"
    ] = np.arange(
        1,
        up.shape[0] + 1,
    )

    top_tables.append(
        up
    )

    top20_up_tables.append(
        up.head(
            20
        ).copy()
    )


    down = sub[
        (
            sub[
                "logfoldchange"
            ]
            < 0
        )
        & (
            sub[
                "score"
            ]
            < 0
        )
    ].copy()

    down = down.sort_values(
        [
            "padj",
            "pval",
            "score",
            "logfoldchange",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
    )

    down = down.head(
        N_TOP_DIRECTION
    )

    down[
        "direction"
    ] = "down"

    down[
        "direction_rank"
    ] = np.arange(
        1,
        down.shape[0] + 1,
    )

    top_tables.append(
        down
    )


    filtered = sub[
        (
            sub[
                "padj"
            ]
            < 0.05
        )
        & (
            sub[
                "logfoldchange"
            ].abs()
            >= 0.25
        )
    ].copy()

    filtered = filtered.sort_values(
        [
            "padj",
            "pval",
            "score",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )

    filtered_tables.append(
        filtered
    )


top100_up_down = pd.concat(
    top_tables,
    axis=0,
    ignore_index=True,
)

top20_up = pd.concat(
    top20_up_tables,
    axis=0,
    ignore_index=True,
)

filtered_de = pd.concat(
    filtered_tables,
    axis=0,
    ignore_index=True,
)


top100_up_down = (
    top100_up_down
    .sort_values(
        [
            "cluster",
            "direction",
            "direction_rank",
        ],
        key=lambda series: (
            series.map(
                natural_cluster_key
            )
            if series.name
            == "cluster"
            else series
        ),
    )
)

top20_up = (
    top20_up
    .sort_values(
        [
            "cluster",
            "direction_rank",
        ],
        key=lambda series: (
            series.map(
                natural_cluster_key
            )
            if series.name
            == "cluster"
            else series
        ),
    )
)


write_tsv_replace(
    top100_up_down,
    DE_TOP100_UP_DOWN_TSV,
)

write_tsv_replace(
    top20_up,
    DE_TOP20_UP_TSV,
)

write_tsv_replace(
    filtered_de,
    DE_FILTERED_TSV,
)


cluster0 = (
    all_de.loc[
        all_de[
            "cluster"
        ]
        == "0"
    ]
    .sort_values(
        "rank_scanpy"
    )
    .head(
        5
    )
)


observed_cluster0_top5 = (
    cluster0[
        "gene"
    ]
    .astype(str)
    .tolist()
)

observed_cluster0_scores = (
    cluster0[
        "score"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


audit_rows = [
    {
        "checkpoint":
            "all_de_shape",
        "historical":
            str(
                HISTORICAL_ALL_DE_SHAPE
            ),
        "current":
            str(
                all_de.shape
            ),
        "match":
            all_de.shape
            == HISTORICAL_ALL_DE_SHAPE,
    },
    {
        "checkpoint":
            "top100_shape",
        "historical":
            str(
                HISTORICAL_TOP100_SHAPE
            ),
        "current":
            str(
                top100_up_down.shape
            ),
        "match":
            top100_up_down.shape
            == HISTORICAL_TOP100_SHAPE,
    },
    {
        "checkpoint":
            "top20_shape",
        "historical":
            str(
                HISTORICAL_TOP20_SHAPE
            ),
        "current":
            str(
                top20_up.shape
            ),
        "match":
            top20_up.shape
            == HISTORICAL_TOP20_SHAPE,
    },
    {
        "checkpoint":
            "filtered_shape",
        "historical":
            str(
                HISTORICAL_FILTERED_SHAPE
            ),
        "current":
            str(
                filtered_de.shape
            ),
        "match":
            filtered_de.shape
            == HISTORICAL_FILTERED_SHAPE,
    },
    {
        "checkpoint":
            "cluster0_top5_genes",
        "historical":
            ",".join(
                HISTORICAL_CLUSTER0_TOP5
            ),
        "current":
            ",".join(
                observed_cluster0_top5
            ),
        "match":
            observed_cluster0_top5
            == HISTORICAL_CLUSTER0_TOP5,
    },
    {
        "checkpoint":
            "cluster0_top5_scores",
        "historical":
            ",".join(
                f"{x:.6f}"
                for x
                in HISTORICAL_CLUSTER0_TOP5_SCORES
            ),
        "current":
            ",".join(
                f"{x:.6f}"
                for x
                in observed_cluster0_scores
            ),
        "match":
            bool(
                np.allclose(
                    observed_cluster0_scores,
                    HISTORICAL_CLUSTER0_TOP5_SCORES,
                    rtol=0,
                    atol=0.05,
                )
            ),
    },
]


repro_audit = pd.DataFrame(
    audit_rows
)

write_tsv_replace(
    repro_audit,
    DE_REPRO_AUDIT_TSV,
)

display(
    repro_audit
)


critical_checks = {
    "top100_shape":
        top100_up_down.shape
        == HISTORICAL_TOP100_SHAPE,

    "top20_shape":
        top20_up.shape
        == HISTORICAL_TOP20_SHAPE,

    "filtered_shape":
        filtered_de.shape
        == HISTORICAL_FILTERED_SHAPE,

    "cluster0_top5_genes":
        observed_cluster0_top5
        == HISTORICAL_CLUSTER0_TOP5,

    "cluster0_top5_scores":
        bool(
            np.allclose(
                observed_cluster0_scores,
                HISTORICAL_CLUSTER0_TOP5_SCORES,
                rtol=0,
                atol=0.05,
            )
        ),
}


if not all(
    critical_checks.values()
):
    raise RuntimeError(
        "Historical marker-DE checkpoints were not fully "
        "reproduced. Stop before annotation and inspect "
        f"{DE_REPRO_AUDIT_TSV}."
    )


print(
    "\n[HISTORICAL MARKER-DE CHECKPOINTS REPRODUCED]"
)

print(
    "Top up/down rows:",
    top100_up_down.shape,
)

print(
    "Filtered rows:",
    filtered_de.shape,
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top100_up_down.tsv shape=(3600, 12)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_top20_up.tsv shape=(360, 12)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_leiden_scvi_main_HVG8000_r0_6_wilcoxon_filtered_padj005_logfc025.tsv shape=(192882, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_historical_reproducibility_audit.tsv shape=(6, 4)


,checkpoint,historical,current,match
0,all_de_shape,"(273168, 11)","(273168, 11)",True
1,top100_shape,"(3600, 12)","(3600, 12)",True
2,top20_shape,"(360, 12)","(360, 12)",True
3,filtered_shape,"(192882, 11)","(192882, 11)",True
4,cluster0_top5_genes,"PTPRZ1,BCAN,OLIG1,NOVA1,PTN","PTPRZ1,BCAN,OLIG1,NOVA1,PTN",True
5,cluster0_top5_scores,"307.749878,288.052246,280.849701,271.248901,26...","307.749878,288.052246,280.849701,271.248901,26...",True



[HISTORICAL MARKER-DE CHECKPOINTS REPRODUCED]
Top up/down rows: (3600, 12)
Filtered rows: (192882, 11)


In [9]:
# =========================
# 8) BUILD SCANPY-COMPATIBLE rank_genes_groups .uns
# =========================

def structured_from_ranked(
    frames,
    dtype,
):
    dataframe = pd.DataFrame(
        {
            cluster:
                frames[
                    cluster
                ]
            for cluster
            in cluster_order
        }
    )

    return dataframe.to_records(
        index=False,
        column_dtypes=dtype,
    )


ranked_names = {}
ranked_scores = {}
ranked_logfc = {}
ranked_pvals = {}
ranked_padj = {}


for cluster in cluster_order:
    table = (
        all_de.loc[
            all_de[
                "cluster"
            ]
            == cluster
        ]
        .sort_values(
            "rank_scanpy"
        )
    )

    ranked_names[
        cluster
    ] = table[
        "gene"
    ].astype(str).to_numpy()

    ranked_scores[
        cluster
    ] = table[
        "score"
    ].to_numpy(
        dtype=np.float32
    )

    ranked_logfc[
        cluster
    ] = table[
        "logfoldchange"
    ].to_numpy(
        dtype=np.float32
    )

    ranked_pvals[
        cluster
    ] = table[
        "pval"
    ].to_numpy(
        dtype=np.float64
    )

    ranked_padj[
        cluster
    ] = table[
        "padj"
    ].to_numpy(
        dtype=np.float64
    )


rank_uns = {
    "params": {
        "groupby":
            MAIN_CLUSTER_KEY,
        "reference":
            "rest",
        "method":
            "wilcoxon",
        "use_raw":
            False,
        "layer":
            None,
        "corr_method":
            "benjamini-hochberg",
    },

    "names":
        structured_from_ranked(
            ranked_names,
            "O",
        ),

    "scores":
        structured_from_ranked(
            ranked_scores,
            "float32",
        ),

    "logfoldchanges":
        structured_from_ranked(
            ranked_logfc,
            "float32",
        ),

    "pvals":
        structured_from_ranked(
            ranked_pvals,
            "float64",
        ),

    "pvals_adj":
        structured_from_ranked(
            ranked_padj,
            "float64",
        ),

    "pts":
        pd.DataFrame(
            {
                cluster:
                    np.asarray(
                        pts_mm[
                            group_index
                        ],
                        dtype=np.float64,
                    )
                for group_index, cluster
                in enumerate(
                    cluster_order
                )
            },
            index=var_names.copy(),
        ),

    "pts_rest":
        pd.DataFrame(
            {
                cluster:
                    np.asarray(
                        pts_rest_mm[
                            group_index
                        ],
                        dtype=np.float64,
                    )
                for group_index, cluster
                in enumerate(
                    cluster_order
                )
            },
            index=var_names.copy(),
        ),
}


with h5py.File(
    MARKER_READY_BUILDING_H5AD,
    "r+",
) as handle:

    if "uns" not in handle:
        write_elem(
            handle,
            "uns",
            {},
        )

    if RANK_KEY in handle[
        "uns"
    ]:
        del handle[
            "uns"
        ][
            RANK_KEY
        ]

    write_elem(
        handle[
            "uns"
        ],
        RANK_KEY,
        rank_uns,
    )

    handle.flush()


print(
    "[PATCHED rank_genes_groups UNS]",
    RANK_KEY,
)

[PATCHED rank_genes_groups UNS] rank_genes_groups_leiden_scvi_main_HVG8000_r0_6_wilcoxon


In [10]:
# =========================
# 9) MEMORY-SAFE QC PLOTS
# =========================

def load_normalized_gene_subset(
    genes,
):
    genes = list(
        dict.fromkeys(
            str(gene)
            for gene
            in genes
            if str(gene)
            in var_set
        )
    )

    target_index = {
        gene:
            index
        for index, gene
        in enumerate(
            genes
        )
    }

    selected_global = {
        int(
            var_names.get_loc(
                gene
            )
        ):
            target_index[
                gene
            ]
        for gene in genes
    }


    row_parts = []
    col_parts = []
    value_parts = []


    block_to_targets = {}

    for global_gene_index, target_column in selected_global.items():
        block_id = (
            global_gene_index
            // GENE_BLOCK_SIZE
        )

        local_column = (
            global_gene_index
            - block_id
            * GENE_BLOCK_SIZE
        )

        block_to_targets.setdefault(
            block_id,
            {}
        )[
            local_column
        ] = target_column


    for block_id, local_map in block_to_targets.items():
        path = block_path(
            block_id
        )

        records = np.memmap(
            path,
            dtype=BLOCK_RECORD_DTYPE,
            mode="r",
        )

        local_values = np.asarray(
            records[
                "col"
            ],
            dtype=np.int32,
        )

        wanted_local = np.fromiter(
            local_map.keys(),
            dtype=np.int32,
        )

        mask = np.isin(
            local_values,
            wanted_local,
        )

        if not np.any(
            mask
        ):
            del records
            continue


        selected_rows = np.asarray(
            records[
                "row"
            ][
                mask
            ],
            dtype=np.int32,
        )

        selected_local = (
            local_values[
                mask
            ]
        )

        selected_values = np.asarray(
            records[
                "val"
            ][
                mask
            ],
            dtype=np.float32,
        )


        lookup = np.full(
            GENE_BLOCK_SIZE,
            -1,
            dtype=np.int32,
        )

        for local_column, target_column in local_map.items():
            lookup[
                local_column
            ] = target_column


        selected_target_columns = (
            lookup[
                selected_local
            ]
        )


        row_parts.append(
            selected_rows
        )

        col_parts.append(
            selected_target_columns
        )

        value_parts.append(
            selected_values
        )

        del records
        del local_values
        gc.collect()


    if len(row_parts) == 0:
        matrix = sp.csr_matrix(
            (
                EXPECTED_N_CELLS,
                len(genes),
            ),
            dtype=np.float32,
        )

    else:
        rows_all = np.concatenate(
            row_parts
        )

        cols_all = np.concatenate(
            col_parts
        )

        values_all = np.concatenate(
            value_parts
        )

        matrix = sp.coo_matrix(
            (
                values_all,
                (
                    rows_all,
                    cols_all,
                ),
            ),
            shape=(
                EXPECTED_N_CELLS,
                len(genes),
            ),
            dtype=np.float32,
        ).tocsr()


    return (
        matrix,
        genes,
    )


# UMAP and cluster-size plots need no expression matrix.
umap_adata = ad.AnnData(
    X=None,
    obs=obs.copy(),
)

if x_umap is not None:
    umap_adata.obsm[
        "X_umap"
    ] = x_umap


if "X_umap" in umap_adata.obsm:
    for color in (
        MAIN_CLUSTER_KEY,
        DATASET_COL,
        CONDITION_COL,
    ):
        if color not in umap_adata.obs.columns:
            continue

        sc.pl.umap(
            umap_adata,
            color=color,
            legend_loc="right margin",
            size=2,
            frameon=True,
            show=False,
        )

        figure = plt.gcf()

        output = (
            MARKER_FIG_DIR
            / f"marker_DE_UMAP_{color}.png"
        )

        figure.savefig(
            output,
            dpi=200,
            bbox_inches="tight",
        )

        plt.close(
            figure
        )

        print(
            "[SAVED]",
            output,
        )


figure, axis = plt.subplots(
    figsize=(
        10,
        4,
    )
)

plot_df = qc.sort_values(
    "cluster",
    key=lambda series: series.map(
        natural_cluster_key
    ),
)

axis.bar(
    plot_df[
        "cluster"
    ].astype(str),
    plot_df[
        "n_cells"
    ],
)

axis.set_xlabel(
    "Cluster"
)

axis.set_ylabel(
    "Number of cells"
)

axis.set_title(
    f"Cluster sizes — {MAIN_CLUSTER_KEY}"
)

plt.xticks(
    rotation=45
)

output = (
    MARKER_FIG_DIR
    / "marker_DE_cluster_sizes.png"
)

figure.savefig(
    output,
    dpi=200,
    bbox_inches="tight",
)

plt.close(
    figure
)

print(
    "[SAVED]",
    output,
)


# Top 3 upregulated genes per cluster.
plot_genes = []

for cluster, sub in top20_up.groupby(
    "cluster",
    sort=False,
):
    genes = (
        sub
        .sort_values(
            "direction_rank"
        )[
            "gene"
        ]
        .head(
            N_TOP_PLOT_PER_CLUSTER
        )
        .astype(str)
        .tolist()
    )

    plot_genes.extend(
        genes
    )


plot_genes = list(
    dict.fromkeys(
        plot_genes
    )
)


known_panel_genes = [
    gene
    for genes
    in marker_panels.values()
    for gene in genes
    if gene in var_set
]


plot_union = list(
    dict.fromkeys(
        plot_genes
        + known_panel_genes
    )
)


plot_matrix, plot_union = (
    load_normalized_gene_subset(
        plot_union
    )
)


plot_adata = ad.AnnData(
    X=plot_matrix,
    obs=obs.copy(),
    var=pd.DataFrame(
        index=plot_union,
    ),
)

plot_adata.uns[
    "log1p"
] = {
    "base":
        None
}

if x_umap is not None:
    plot_adata.obsm[
        "X_umap"
    ] = x_umap


if len(
    plot_genes
):
    sc.pl.dotplot(
        plot_adata,
        var_names=[
            gene
            for gene
            in plot_genes
            if gene
            in plot_adata.var_names
        ],
        groupby=MAIN_CLUSTER_KEY,
        standard_scale="var",
        dendrogram=False,
        show=False,
    )

    figure = plt.gcf()

    output = (
        MARKER_FIG_DIR
        / "marker_DE_top3_up_per_cluster_dotplot.png"
    )

    figure.savefig(
        output,
        dpi=220,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )

    print(
        "[SAVED]",
        output,
    )


marker_panels_present = {}

for panel, genes in marker_panels.items():
    present = [
        gene
        for gene
        in genes
        if gene
        in plot_adata.var_names
    ]

    if present:
        marker_panels_present[
            panel
        ] = present


if marker_panels_present:
    sc.pl.dotplot(
        plot_adata,
        var_names=marker_panels_present,
        groupby=MAIN_CLUSTER_KEY,
        standard_scale="var",
        # Historical notebook requested dendrogram=True, which implicitly
        # triggered a full-expression PCA. We disable only that QC-plot
        # ordering step to keep this notebook memory safe.
        dendrogram=False,
        show=False,
    )

    figure = plt.gcf()

    output = (
        MARKER_FIG_DIR
        / "marker_DE_known_marker_panel_dotplot.png"
    )

    figure.savefig(
        output,
        dpi=220,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )

    print(
        "[SAVED]",
        output,
    )


del plot_adata
del plot_matrix
del umap_adata
gc.collect()

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\marker_DE_HVG8000_r0_6_pre_annotation\marker_DE_UMAP_leiden_scvi_main.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\marker_DE_HVG8000_r0_6_pre_annotation\marker_DE_UMAP_core_dataset.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\marker_DE_HVG8000_r0_6_pre_annotation\marker_DE_UMAP_condition.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\marker_DE_HVG8000_r0_6_pre_annotation\marker_DE_cluster_sizes.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\marker_DE_HVG8000_r0_6_pre_annotation\marker_DE_top3_up_per_cluster_dotplot.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\figures\marker_DE_HVG8000_r0_6_pre_annotation\marker_DE_known_marker_panel_dotplot.png


45

In [11]:
# =========================
# 10) VALIDATE + PUBLISH MARKER-READY H5AD
# =========================

check = sc.read_h5ad(
    MARKER_READY_BUILDING_H5AD,
    backed="r",
)

try:
    print(
        "Building object:",
        check,
    )

    if check.shape != (
        EXPECTED_N_CELLS,
        EXPECTED_N_GENES,
    ):
        raise AssertionError(
            "Marker-ready shape changed."
        )

    if "counts" not in check.layers:
        raise AssertionError(
            "Marker-ready object lost the raw counts layer."
        )

    if RANK_KEY not in check.uns:
        raise AssertionError(
            "Marker-ready object is missing rank_genes_groups results."
        )

    if MAIN_CLUSTER_KEY not in check.obs.columns:
        raise AssertionError(
            "Marker-ready object lost canonical clusters."
        )

    validated_counts = (
        check.obs[
            MAIN_CLUSTER_KEY
        ]
        .astype(str)
        .value_counts()
    )

    if validated_counts.shape[0] != EXPECTED_N_CLUSTERS:
        raise AssertionError(
            "Marker-ready object is not 18 clusters."
        )

    print(
        "Validated shape:",
        check.shape,
    )

    print(
        "Validated counts layer:",
        "counts"
        in check.layers,
    )

    print(
        "Validated rank key:",
        RANK_KEY,
    )

    print(
        "Validated canonical clusters:",
        validated_counts.shape[0],
    )

finally:
    check.file.close()


if MARKER_READY_H5AD.exists():
    MARKER_READY_H5AD.unlink()


os.replace(
    MARKER_READY_BUILDING_H5AD,
    MARKER_READY_H5AD,
)


print(
    "\n[PUBLISHED MARKER-READY H5AD]"
)

print(
    MARKER_READY_H5AD
)

Building object: AnnData object with n_obs × n_vars = 351427 × 15176 backed at 'G:\\Repository\\glioma-cnv-single-cell-atlas\\data\\processed\\integrated\\GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_DE_ready.building.h5ad'
    obs: 'sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'dataset_id', 'study_id', 'source_dataset', 'source_file', 'concat_key', 'patient_id', 'condition', 'batch_dataset', 'batch_sample', 'batch_patient', 'original_cell_barcode', 'diagnosis', 'disease_stage', 'tumor_status', 'platform', 'technology', 'study_batch', 'sample_batch', 'scvi_batch', 'analysis_group_major', 'analysis_group_detail', 'ready_for_integration', 'cell_id_original', 'leiden_scvi_0.1', 'leiden_scvi_0.2', 'leiden_scvi_0.3', 'leiden_scvi_0.4', 'leiden_scvi_0.5', 'leiden_scvi_HVG8000_r0_1', 'leiden_scvi_HVG8000_r0_15', 'leiden_scvi_HVG8000_r0_2', 'leiden_sc

In [12]:
# =========================
# 11) FINAL MANIFEST
# =========================

manifest = pd.DataFrame(
    [
        {
            "artifact":
                "input_h5ad",
            "path":
                str(
                    FULLGENE_INPUT_H5AD
                ),
            "exists":
                FULLGENE_INPUT_H5AD.exists(),
        },
        {
            "artifact":
                "marker_ready_h5ad",
            "path":
                str(
                    MARKER_READY_H5AD
                ),
            "exists":
                MARKER_READY_H5AD.exists(),
        },
        {
            "artifact":
                "readiness_report",
            "path":
                str(
                    DE_READINESS_TSV
                ),
            "exists":
                DE_READINESS_TSV.exists(),
        },
        {
            "artifact":
                "cluster_qc",
            "path":
                str(
                    CLUSTER_QC_TSV
                ),
            "exists":
                CLUSTER_QC_TSV.exists(),
        },
        {
            "artifact":
                "cluster_composition",
            "path":
                str(
                    CLUSTER_COMPOSITION_TSV
                ),
            "exists":
                CLUSTER_COMPOSITION_TSV.exists(),
        },
        {
            "artifact":
                "marker_panel_presence",
            "path":
                str(
                    MARKER_PANEL_PRESENCE_TSV
                ),
            "exists":
                MARKER_PANEL_PRESENCE_TSV.exists(),
        },
        {
            "artifact":
                "all_de",
            "path":
                str(
                    DE_ALL_TSV
                ),
            "exists":
                DE_ALL_TSV.exists(),
        },
        {
            "artifact":
                "top100_up_down",
            "path":
                str(
                    DE_TOP100_UP_DOWN_TSV
                ),
            "exists":
                DE_TOP100_UP_DOWN_TSV.exists(),
        },
        {
            "artifact":
                "top20_up",
            "path":
                str(
                    DE_TOP20_UP_TSV
                ),
            "exists":
                DE_TOP20_UP_TSV.exists(),
        },
        {
            "artifact":
                "filtered_de",
            "path":
                str(
                    DE_FILTERED_TSV
                ),
            "exists":
                DE_FILTERED_TSV.exists(),
        },
        {
            "artifact":
                "historical_reproducibility_audit",
            "path":
                str(
                    DE_REPRO_AUDIT_TSV
                ),
            "exists":
                DE_REPRO_AUDIT_TSV.exists(),
        },
        {
            "artifact":
                "figures_dir",
            "path":
                str(
                    MARKER_FIG_DIR
                ),
            "exists":
                MARKER_FIG_DIR.exists(),
        },
        {
            "artifact":
                "normalization_checkpoint",
            "path":
                str(
                    NORMALIZATION_PROGRESS_JSON
                ),
            "exists":
                NORMALIZATION_PROGRESS_JSON.exists(),
        },
        {
            "artifact":
                "de_checkpoint",
            "path":
                str(
                    DE_PROGRESS_JSON
                ),
            "exists":
                DE_PROGRESS_JSON.exists(),
        },
    ]
)


write_tsv_replace(
    manifest,
    DE_OUTPUT_MANIFEST_TSV,
)

display(
    manifest
)


print(
    "\nDONE — historical marker-DE semantics reproduced "
    "with memory-safe blocked execution."
)

print(
    "All DE:",
    all_de.shape,
)

print(
    "Top100 up/down:",
    top100_up_down.shape,
)

print(
    "Top20 up:",
    top20_up.shape,
)

print(
    "Filtered:",
    filtered_de.shape,
)

print(
    "Cluster 0 top five:",
    observed_cluster0_top5,
)

print(
    "Marker-ready output:",
    MARKER_READY_H5AD,
)

[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\marker_DE_HVG8000_r0_6_output_manifest.tsv shape=(14, 3)


,artifact,path,exists
0,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
1,marker_ready_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
2,readiness_report,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
3,cluster_qc,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
4,cluster_composition,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
5,marker_panel_presence,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
6,all_de,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
7,top100_up_down,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
8,top20_up,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
9,filtered_de,G:\Repository\glioma-cnv-single-cell-atlas\met...,True



DONE — historical marker-DE semantics reproduced with memory-safe blocked execution.
All DE: (273168, 11)
Top100 up/down: (3600, 12)
Top20 up: (360, 12)
Filtered: (192882, 11)
Cluster 0 top five: ['PTPRZ1', 'BCAN', 'OLIG1', 'NOVA1', 'PTN']
Marker-ready output: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_DE_ready.h5ad


## Expected successful completion

The historical executed notebook reported:

```text
all_de            (273168, 11)
top100 up/down      (3600, 12)
top20 up             (360, 12)
filtered           (192882, 11)

cluster 0 top five:
PTPRZ1, BCAN, OLIG1, NOVA1, PTN

marker-panel genes present: 79 / 81
missing: MS4A1, JCHAIN
```

The final marker-ready H5AD contains:

- log-normalized full-gene `X`;
- raw `layers["counts"]`;
- canonical historical 18-cluster metadata;
- canonical `X_scVI` and `X_umap` inherited from notebook 06;
- Scanpy-compatible `rank_genes_groups` results under the historical `RANK_KEY`.

Both the normalization/cache stage and the DE gene-block stage are resumable.
Do not delete `E:\glioma-cnv-scratch\main_analysis_07` until this notebook has
completed and the outputs have been validated.